# 17-01　联合训练与三种推理接口

第 17 章讨论 UWM 如何让动作与未来画面共同参与训练。这份实践只保留其中最容易核对的关系：**当前状态、实际动作、执行后的状态**来自同一条记录。教学数据是一维推物，不含相机图像、视频 VAE 或 Diffusion Transformer，因此下方的数字和曲线都不能当作 UWM 的实验结果。

请按顺序运行：先看一条记录提供的两份答案，再分别给动作和未来加噪，最后用一个小型共享网络练习同时估计两份已知噪声。这里的“未来”是物体位置；论文中的未来则是图像潜变量。每段输出后都附有读图说明。

## 1. 一条推动记录为什么有两份答案

设物体推动前位于数轴上的 $c$，机器人实际执行位移 $a$；教学环境规定下一位置为 $y=c+0.7a$。于是每条记录都是 $(c,a,y)$。同一条记录既能回答“在这个起点执行了什么动作”（答案 $a$），也能回答“执行这个动作后到哪里”（答案 $y$）。图中每条细线只连接**同一条记录**的起点与终点，绝不是把不同记录任意配对。请先观察向左与向右动作分别对应怎样的终点。

In [ ]:
import numpy as np  # 导入数组工具，生成可重复的一维推动记录
import matplotlib.pyplot as plt  # 导入绘图工具，画出动作与真实后果
rng = np.random.default_rng(17)  # 固定随机种子，保证每次从头运行得到同一批教学记录
current = rng.uniform(-0.8, 0.8, size=12)  # 为十二条记录分别抽取推动前的物体位置
action = rng.uniform(-0.9, 0.9, size=12)  # 为每条记录抽取实际执行的有符号位移
future = current + 0.7 * action  # 按教学环境的规则计算动作执行后的真实位置
fig, ax = plt.subplots(figsize=(8, 4))  # 建立一幅便于比较起点与终点的图
for index in range(len(current)):  # 逐条处理记录，避免把不同记录的点连在一起
    ax.plot([0, 1], [current[index], future[index]], color='#9aaab3', alpha=0.7)  # 用细线连接同一条记录的前后位置
ax.scatter(np.zeros_like(current), current, color='#376381', s=65, label='before')  # 用蓝点表示每条记录的真实起点
ax.scatter(np.ones_like(future), future, color='#bc7843', s=65, label='after')  # 用橙点表示与该动作对应的真实终点
ax.set_xticks([0, 1], ['before action', 'after action'])  # 标出横轴上的执行前与执行后
ax.set_ylabel('object position')  # 说明纵轴表示同一物体的数轴位置
ax.set_title('One interaction gives an action and its consequence')  # 给图像写出要核对的教学问题
ax.grid(axis='y', alpha=0.2)  # 加入浅色参考线方便比较位移方向
ax.legend()  # 显示蓝色与橙色点分别代表的时间角色
plt.show()  # 显示本段实践的第一幅结果图
print(f'Example: current={current[0]:+.2f}, action={action[0]:+.2f}, future={future[0]:+.2f}')  # 打印一条记录供读者核算未来位置

**怎样读结果。** 蓝点是动作前的位置，橙点是动作后的真实位置。细线朝上表示这条记录的动作向右，朝下表示向左；线的长度与动作数值成正比。最后一行打印第一条记录的三个数，可自行核对 $y=c+0.7a$。例如 $a$ 为负时，$y$ 应小于 $c$。若把第一条记录的动作错配到第二条记录的终点，模型即使努力降低误差，也会收到自相矛盾的训练答案。这里的两份答案来自一次交互，而不是网络凭空造出的标签。

## 2. 动作与未来为什么各有自己的噪声进度

从上面第一条记录取出干净动作 $a$ 与真实未来 $y$。我们人为规定四个加噪等级：等级 0 完全保留原数，等级 3 大部分由随机噪声组成。分别设置动作等级 $k_a$ 和未来等级 $k_y$；二者不必相等。下面左右两幅图使用**同一条记录与同一对随机噪声**，只交换加噪等级，因此差别来自进度设置而不是换了样本。图中的灰色虚线是原值，实心点是带噪输入；训练时真正核对的答案则是事先抽出的噪声本身。

In [ ]:
alpha_bars = np.array([1.0, 0.80, 0.40, 0.08])  # 规定四个等级的干净信号保留比例
clean_pair = np.array([action[0], future[0]])  # 取同一条记录的动作与动作后的真实位置
known_noise = rng.normal(size=2)  # 分别抽取两份之后要让网络估计的已知噪声
level_pairs = [(3, 1), (1, 3)]  # 设置动作重噪且未来轻噪，以及反过来的两种情况
fig, axes = plt.subplots(1, 2, figsize=(9, 3.8), sharey=True)  # 并排显示两个独立进度组合
for ax, levels in zip(axes, level_pairs):  # 对每一种进度组合构造一次带噪输入
    kept = alpha_bars[list(levels)]  # 读取动作与未来各自对应的信号保留比例
    noisy = np.sqrt(kept) * clean_pair + np.sqrt(1 - kept) * known_noise  # 用与正文一致的加噪公式形成两个输入
    ax.scatter([0, 1], clean_pair, marker='_', s=550, color='#687c89', label='clean')  # 画出未加噪的两项真实数值
    ax.scatter([0, 1], noisy, s=105, color=['#bc7843', '#457d74'], label='noisy')  # 画出分别加噪后的动作与未来
    ax.set_xticks([0, 1], ['action', 'future'])  # 标明左侧数值是动作、右侧数值是未来位置
    ax.set_title(f'action level={levels[0]}, future level={levels[1]}')  # 在图题中写明两条支路各自的等级
    ax.grid(axis='y', alpha=0.2)  # 加入浅参考线以比较点和原值的偏离
axes[0].set_ylabel('one recorded value')  # 说明纵轴是这条真实记录中的数值
axes[1].legend()  # 解释短横线代表原值而圆点代表带噪输入
plt.show()  # 显示两个独立进度组合的对照图
print(f'Known noise to predict: action={known_noise[0]:+.3f}, future={known_noise[1]:+.3f}')  # 打印两项监督答案，说明它们不是网络输入

**怎样读结果。** 左图动作等级为 3，未来等级为 1；右图把这两个等级交换。圆点距离短横线越远，表示本次抽到的噪声在该数值上影响越明显，但它不保证等级更大时每一个样本的绝对误差都严格更大：随机噪声恰好也可能接近原值。关键是两条支路可以分别设置难度。最后打印的两项噪声是我们加进去时就知道的训练答案；它们**没有**作为待训练网络的额外输入。这里的等级是扩散训练进度，不是推杆轨迹中的第几秒。

## 3. 一个共享网络怎样同时接受两项训练信号

下面训练一个小型多层感知机（MLP），**仅用于练习 UWM 的联合噪声目标**。每次迭代重新抽取一批当前状态、实际动作、两个独立噪声等级及两份高斯噪声。网络输入是当前状态、两项带噪数值和两个等级；输出是两个数，分别估计动作噪声与未来噪声。它没有图像编码器，也没有 Transformer，因此不能据此推断大型 UWM 的生成质量。

图像给出总误差和两项误差的移动平均值。请看两条分项曲线是否都低于初期，而不要仅凭总误差变小就断言模型学会了安全推动或视觉预测。总误差由两项相加，数值通常比单项更大。

In [ ]:
import torch  # 导入张量与自动求导工具，训练教学用共享网络
from torch import nn  # 导入全连接层和激活函数等网络组件
torch.manual_seed(17)  # 固定网络初始化和训练抽样，方便复现曲线
torch.set_num_threads(2)  # 限制线程数量，让普通 Colab CPU 也能较快运行
schedule = torch.tensor([1.0, 0.80, 0.40, 0.08])  # 使用上一段相同的四级加噪日程
model = nn.Sequential(nn.Linear(5, 48), nn.ReLU(), nn.Linear(48, 48), nn.ReLU(), nn.Linear(48, 2))  # 用共享隐藏层和双数值输出估计两份噪声
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)  # 创建调整共享网络参数的优化器
history = []  # 准备记录每一步的总误差和两项分误差
for step in range(360):  # 重复生成小批次并完成三百六十次参数更新
    c = torch.rand(256, 1) * 1.6 - 0.8  # 抽取一批动作执行前的物体位置
    a = torch.rand(256, 1) * 1.8 - 0.9  # 抽取同批记录中的实际动作位移
    y = c + 0.7 * a  # 用教学环境规则得到每条记录的真实未来位置
    ka = torch.randint(0, 4, (256, 1))  # 独立抽取每条记录的动作噪声等级
    ky = torch.randint(0, 4, (256, 1))  # 独立抽取每条记录的未来噪声等级
    ea = torch.randn_like(a)  # 抽取已知的动作噪声监督答案
    ey = torch.randn_like(y)  # 抽取已知的未来噪声监督答案
    aa = torch.sqrt(schedule[ka]) * a + torch.sqrt(1 - schedule[ka]) * ea  # 构造带噪动作输入
    yy = torch.sqrt(schedule[ky]) * y + torch.sqrt(1 - schedule[ky]) * ey  # 构造带噪未来输入
    features = torch.cat([c, aa, yy, ka.float() / 3, ky.float() / 3], dim=1)  # 拼接网络可见的五项输入，不拼接真实噪声
    prediction = model(features)  # 让共享网络一次给出两项噪声估计
    loss_a = ((prediction[:, :1] - ea) ** 2).mean()  # 核对动作噪声估计与已知答案的平方误差
    loss_y = ((prediction[:, 1:] - ey) ** 2).mean()  # 核对未来噪声估计与已知答案的平方误差
    loss = loss_a + loss_y  # 把两项训练要求相加供本次更新使用
    optimizer.zero_grad()  # 清除上一次更新遗留的参数梯度
    loss.backward()  # 根据总误差计算共享网络参数的梯度
    optimizer.step()  # 按梯度实际更新网络参数
    history.append([loss.item(), loss_a.item(), loss_y.item()])  # 保存本次的三项数值供运行后观察
history_array = np.asarray(history)  # 把误差记录转成便于绘图的数组
window = 18  # 规定移动平均窗口以减小随机批次造成的曲线抖动
smoothed = np.stack([np.convolve(history_array[:, column], np.ones(window) / window, mode='valid') for column in range(3)], axis=1)  # 分别平滑总误差与两项分误差
fig, ax = plt.subplots(figsize=(8, 4))  # 建立单图比较三条误差曲线
ax.plot(smoothed[:, 0], color='#374f68', label='total')  # 画出两项相加后的训练误差
ax.plot(smoothed[:, 1], color='#bc7843', label='action noise')  # 画出动作噪声估计误差
ax.plot(smoothed[:, 2], color='#457d74', label='future noise')  # 画出未来噪声估计误差
ax.set_xlabel('training update')  # 说明横轴代表参数更新次数而非机器人时间
ax.set_ylabel('mean squared error')  # 说明纵轴是本批次噪声估计的均方误差
ax.set_title('One shared model, two known-noise targets')  # 标出共享模型对应的两份答案
ax.grid(alpha=0.2)  # 加入浅网格方便比较训练初期与末期
ax.legend()  # 显示三条曲线各代表哪一项误差
plt.show()  # 显示共享网络的训练结果
print(f'Last 18 updates: action loss={smoothed[-1, 1]:.3f}, future loss={smoothed[-1, 2]:.3f}')  # 打印末段两项误差便于逐项核对

**怎样读结果。** 深色总误差大致是橙色动作误差与绿色未来误差之和；两项误差来自**同一个网络的不同输出位置**。曲线开始下降说明网络能利用当前值、带噪动作、带噪未来及各自噪声等级估计本次加入的噪声。如果未来误差下降而动作误差没有下降，总误差也可能下降，所以必须分项检查。这里没有把噪声估计反复用于采样，更没有生成视频或执行机械臂；数值低不代表机器人任务成功。

试着把上方代码里的 `360` 改为 `60`，重新从头运行，观察两项误差是否还一样稳定；这能帮助区分训练曲线与真实生成能力。

## 4. 三种用法到底固定了什么、生成了什么

最后把第 17 章图 17-3 的三种用法画成一张并列表。每行是**一次不同的模型调用方式**，不是先后执行的三个步骤；每列是当前状态、动作、未来的位置。蓝色表示给定，橙色表示要生成，灰色表示完全噪声占位且本次不把它生成为结果。若要用网络真正生成，需要再编写扩散采样循环；下面只检查接口角色，不伪称前面的小网络已经产生了可执行动作或视频。

In [ ]:
from matplotlib.patches import Rectangle  # 导入矩形图元，给每种输入输出角色着色
roles = [['given', 'generate', 'unused'], ['given', 'given', 'generate'], ['given', 'generate', 'given']]  # 逐行定义策略、正向和逆向用法
names = ['policy: action', 'forward: future', 'inverse: action']  # 给三种使用方式标注要回答的问题
columns = ['current', 'action', 'future']  # 固定三列所代表的数据角色
colors = {'given': '#dceaf2', 'generate': '#f7dec4', 'unused': '#eceff1'}  # 用浅蓝、浅橙和浅灰区分角色
fig, ax = plt.subplots(figsize=(9, 3.5))  # 建立一张三行三列的接口对照图
for row, row_roles in enumerate(roles):  # 依次画出三种用法的整行角色
    for column, role in enumerate(row_roles):  # 在当前用法中依次处理三类数据
        ax.add_patch(Rectangle((column, row), 1, 1, facecolor=colors[role], edgecolor='white', linewidth=3))  # 按角色为对应格子填色
        ax.text(column + 0.5, row + 0.5, role, ha='center', va='center', fontsize=12)  # 在格子中直接写出角色名称
ax.set_xlim(0, 3)  # 让横向范围恰好容纳当前、动作、未来三列
ax.set_ylim(3, 0)  # 让策略用法排在图的第一行
ax.set_xticks([0.5, 1.5, 2.5], columns)  # 为三列添加数据名称
ax.set_yticks([0.5, 1.5, 2.5], names)  # 为三行添加对应的推理问题
ax.tick_params(length=0)  # 隐藏不表示数值尺度的刻度短线
for spine in ax.spines.values():  # 清理这张角色表周围多余的坐标轴边框
    spine.set_visible(False)  # 关闭当前边框以突出三行角色关系
ax.set_title('Which variable is given, generated, or unused?')  # 提醒读者核对的是角色而非先后顺序
plt.show()  # 显示三种调用方式的角色对照图

**怎样读结果。** 第一行只有动作格为橙色：当前状态已知，未来格只是占位；因此联合训练的模型可以只求动作，**不必先生成视频**。第二行把动作格改为蓝色，未来格变成橙色，对应“已给定动作，求执行后的状态”。第三行给定当前与指定未来，橙色动作格才需要生成，回答“可能用什么动作到达那里”。这三行共享训练得到的参数，但所需已知条件不同。图中的未来只是教学数据的一个位置值；若换成 UWM，未来支路处理的是画面潜变量。此处没有采样循环，所以这幅图不是三次模型推理的结果，而是推理前必须明确的接口约定。